# 02 - Data Preparation

## Purpose

The purpose of this notebook is to prepare the Liberty Mutual Commercial Property Fire-Peril Loss Cost dataset for statistical modelling. This means cleaning the raw data, handling missing values, transforming and scaling numeric variables, encoding categorical predictors and constructing a reproducible preprocessing pipeline that is suitable for both generalized linear models and machine-learning algorithms.

## Methodology

Data preparation involves cleaning and transforming the raw dataset to make it suitable for statistical modelling. Missing values are handled by using imputation strategies appropriate for skewed insurance data. Numeric variables are scaled and categorical variables are encoded to support both generalized linear models and machine-learning algorithms.

In [37]:
# ======================================================
# CIND820 - Keystone Project
# Data Analysis and Visualization: Fire Peril Loss Cost (Liberty Mutual Insurance)
# Author: Debra Allen 
# ======================================================

# Call Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

## Data Loading

In [38]:
# ======================================================
# Load the dataset and display basic information
# ======================================================

def find_repo_root(start_path=None, marker_files=("'git", "README.md")):
    """
    Walk up the directory to find the repository root.
    The root is identified by the presence of marker files (e.g., .git, README.md).
    """
    if start_path is None:
        start_path = Path.cwd()
    
    start_path = start_path.resolve()

    for parent in [start_path] + list(start_path.parents):
        for marker in marker_files:
            if (parent / marker).exists():
                return parent
    raise FileNotFoundError("Could not find repo root. Run notebook from inside the repo.")


REPO_ROOT = find_repo_root()
DATA_PATH = REPO_ROOT / "data" / "liberty_train.csv"
print("Repo Root:", REPO_ROOT)
print("Data Path:", DATA_PATH)

# Read the dataset
data = pd.read_csv(DATA_PATH, low_memory=False)

# Display the first few rows of the dataset
print("\nFirst few rows of the dataset:")
data.head()

Repo Root: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820
Data Path: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820\data\liberty_train.csv

First few rows of the dataset:


,id,target,var1,var2,var3,var4,var5,var6,var7,var8,...,weatherVar227,weatherVar228,weatherVar229,weatherVar230,weatherVar231,weatherVar232,weatherVar233,weatherVar234,weatherVar235,weatherVar236
0,1,0.0,Z,Z,Z,N1,Z,Z,3,1,...,0.170351,0.0,0.00000,0.000000,1.117353,1.215303,0.112556,2.355737,0.404655,0.138667
1,2,0.0,Z,Z,Z,C1,Z,Z,3,2,...,0.266173,0.0,27.50823,0.000000,3.828979,1.036739,0.033052,0.856632,0.231232,0.742199
2,3,0.0,3,Z,4,J3,B,B,2,4,...,0.979517,0.0,0.00000,0.456134,0.098790,1.076535,0.566352,0.696013,0.693695,0.070654
3,4,0.0,3,Z,4,H1,B,Z,3,4,...,0.308761,0.0,0.00000,4.349042,0.401975,0.340631,0.290147,0.000000,0.000000,0.090332
4,5,0.0,Z,Z,Z,H1,Z,Z,2,4,...,2.171972,0.0,0.00000,0.514990,0.516095,1.016120,1.313732,1.338487,2.948202,0.816485


## Data Preparation

In [39]:
# =======================================================
# Data Preparation for Modeling: Fire Peril Loss Cost
# =======================================================

# Import necessary libraries for data preparation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [40]:
# Use a smaller sample for modeling to avoid memory issues
def find_repo_root(start_path=None, marker_files=("'git", "README.md")):
    """
    Walk up the directory to find the repository root.
    The root is identified by the presence of marker files (e.g., .git, README.md).
    """
    if start_path is None:
        start_path = Path.cwd()
    
    start_path = start_path.resolve()

    for parent in [start_path] + list(start_path.parents):
        for marker in marker_files:
            if (parent / marker).exists():
                return parent
    raise FileNotFoundError("Could not find repo root. Run notebook from inside the repo.")


REPO_ROOT = find_repo_root()
SAMPLE_PATH = REPO_ROOT / "data" / "liberty_train_subset.csv"
print("Repo Root:", REPO_ROOT)
print("Data Path:", SAMPLE_PATH)

# Load a smaller sample of the dataset for modeling
model_data = pd.read_csv(SAMPLE_PATH, low_memory=False)

# Display basic information about the dataset
model_data.info()


Repo Root: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820
Data Path: C:\Users\uni_f\Downloads\LaTex\TMU-CIND820\data\liberty_train_subset.csv
<class 'pandas.DataFrame'>
RangeIndex: 113015 entries, 0 to 113014
Columns: 302 entries, id to weatherVar236
dtypes: float64(291), int64(1), str(10)
memory usage: 260.4 MB


In [17]:
# Verify that all categorical variables are strings
for col in model_data.select_dtypes(include=['object']).columns:
    model_data[col] = model_data[col].astype(str)

print("\nAll categorical variables have been converted to strings.")

print("\nDataset Shape:", model_data.shape)


All categorical variables have been converted to strings.

Dataset Shape: (113015, 302)


C:\Users\uni_f\AppData\Local\Temp\ipykernel_21812\4162061741.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in model_data.select_dtypes(include=['object']).columns:


In [18]:
# Define target variable
target = 'target'

# Feature matrix and response
X = model_data.drop(columns=[target])
y = model_data[target]

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("\nCategorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

# Display the number of categorical and numerical columns
print("\nNumber of Categorical Columns:", len(categorical_cols))
print("Number of Numerical Columns:", len(numerical_cols))


Categorical Columns: ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'dummy']
Numerical Columns: ['id', 'var10', 'var11', 'var12', 'var13', 'var14', 'var15', 'var16', 'var17', 'crimeVar1', 'crimeVar2', 'crimeVar3', 'crimeVar4', 'crimeVar5', 'crimeVar6', 'crimeVar7', 'crimeVar8', 'crimeVar9', 'geodemVar1', 'geodemVar2', 'geodemVar3', 'geodemVar4', 'geodemVar5', 'geodemVar6', 'geodemVar7', 'geodemVar8', 'geodemVar9', 'geodemVar10', 'geodemVar11', 'geodemVar12', 'geodemVar13', 'geodemVar14', 'geodemVar15', 'geodemVar16', 'geodemVar17', 'geodemVar18', 'geodemVar19', 'geodemVar20', 'geodemVar21', 'geodemVar22', 'geodemVar23', 'geodemVar24', 'geodemVar25', 'geodemVar26', 'geodemVar27', 'geodemVar28', 'geodemVar29', 'geodemVar30', 'geodemVar31', 'geodemVar32', 'geodemVar33', 'geodemVar34', 'geodemVar35', 'geodemVar36', 'geodemVar37', 'weatherVar1', 'weatherVar2', 'weatherVar3', 'weatherVar4', 'weatherVar5', 'weatherVar6', 'weatherVar7', 'weatherVar8', 'weatherVar9', 

C:\Users\uni_f\AppData\Local\Temp\ipykernel_21812\1606720166.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns.tolist()


## Missing Values

In [ ]:
# =======================================================
# Handle Missing Values and Encode Categorical Variables
# =======================================================

# Categorical columns summary
print("\nSummary Statistics for Categorical Columns:")
print(model_data[categorical_cols].describe().T)
print("\nUnique Value Counts for Categorical Columns:")
for col in categorical_cols:
    print(f"\nUnique value counts for {col}:")
    print(model_data[col].value_counts())

# Percentage of missing values in each column
missing_values = model_data.isnull().mean() * 100
print(f"\nPercentage of Missing Values in Each Column:")
print(missing_values)

# Median for missing numerical values
num_imputer = SimpleImputer(strategy='median')
model_data[numerical_cols] = num_imputer.fit_transform(model_data[numerical_cols])

# Most frequent for missing categorical values
cat_imputer = SimpleImputer(strategy='most_frequent')
model_data[categorical_cols] = cat_imputer.fit_transform(model_data[categorical_cols])

# Verify that missing values are handled
print("\nMissing values after imputation:")
print(model_data.isnull().sum().sort_values(ascending=False).head(20))

# Combine imputation and encoding in a pipeline
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline, numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
],
remainder='drop')


Summary Statistics for Categorical Columns:
        count unique top    freq
var1   113015      6   Z   84432
var2   113015      4   Z  111916
var3   113015      7   Z   98718
var4   113015     43  H1   29105
var5   113015      7   Z  103970
var6   113015      4   Z  106238
var7   113015      9   3   30413
var8   113015      7   2   30450
var9   113015      3   A   73564
dummy  113015      2   A   82166

Unique Value Counts for Categorical Columns:

Unique value counts for var1:
var1
Z    84432
3    14384
2     8324
4     5344
1      518
5       13
Name: count, dtype: int64

Unique value counts for var2:
var2
Z    111916
B       610
C       415
A        74
Name: count, dtype: int64

Unique value counts for var3:
var3
Z    98718
3     5807
4     5035
2     1594
6      843
5      711
1      307
Name: count, dtype: int64

Unique value counts for var4:
var4
H1    29105
D3    13126
M1    10737
O2     8747
Z      5042
D1     4595
G1     4207
E3     3846
C1     2925
I1     2435
A1     2054
N

## Train-Test Split

In [20]:
# =======================================================
# Train-Test Split
# =======================================================

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nTraining observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])


Training observations: 90412
Testing observations: 22603


## Data Preprocessing

In [21]:
# =======================================================
# Data Preprocessing Pipeline
# =======================================================

# Fit prepocessing pipeline on training data and transform both training and testing data
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("\nData preprocessing pipeline has been applied to training and testing data.")

print("\nShape of preprocessed training data:", X_train_prepared.shape)
print("Shape of preprocessed testing data:", X_test_prepared.shape)


Data preprocessing pipeline has been applied to training and testing data.

Shape of preprocessed training data: (90412, 383)
Shape of preprocessed testing data: (22603, 383)


In [22]:
# =======================================================
# Subsample for Modeling Efficiency
# =======================================================

# To further reduce memory usage, take a random subsample of the preprocessed training data for modeling.
sample_size = 5000  # Adjust this number based on memory constraints

if X_train_prepared.shape[0] > sample_size:
    idx = np.random.choice(X_train_prepared.shape[0], sample_size, replace=False)
    X_train_sample = X_train_prepared[idx]
    y_train_sample = y_train.iloc[idx]
    print(f"\nSubsampled {sample_size} observations for modeling.")
else:
    X_train_sample = X_train_prepared
    y_train_sample = y_train
    print("\nTraining data is small enough to use without subsampling.")


Subsampled 5000 observations for modeling.


## Data Preparation Summary

Data preparation involved converting categorical variables to string format, changing missing numeric values with the median and categorical values with the modal categories and standardizing the numeric predictors. Categorical values were one-hot encoded to support machine-learning models. A consistent preprocessing pipeline was applied to training and test data to prevent data leakage. Since there were a large number of variables, a smaller sample was taken so that models could be developed and tested efficiently.